# AutoGluon — CICIoT2023 — Pipeline Binário Organizado

**Objetivo:** treinar um modelo AutoGluon para classificação binária:

- `Benign`
- `Malicious`

Fluxo correto do pipeline:

1. Importar bibliotecas
2. Definir configurações
3. Criar funções auxiliares
4. Localizar arquivos c:/dataset/CSV
5. Carregar o dataset em chunks - dividir um conjunto de dados muito grande em partes menores
6. Tratar e validar o dataset
7. Dividir em treino e teste
8. Treinar o modelo AutoGluon
9. Realizar predições
10. Avaliar métricas
11. Salvar resultados em CSV e PNG


In [1]:
# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

# Execute apenas uma vez, se o AutoGluon ainda não estiver instalado:
# !pip install -U autogluon

import os
import glob
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from autogluon.tabular import TabularDataset, TabularPredictor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

print("Bibliotecas importadas com sucesso.")


Bibliotecas importadas com sucesso.


In [2]:
# ============================================================
# 2. CONFIGURAÇÕES PRINCIPAIS
# ============================================================

DATASET_DIR = Path(r"C:\Dataset\CSV")
RESULTS_DIR = Path(r"C:\Dataset\Resultados\AutoGluon_Binario")
MODEL_DIR = RESULTS_DIR / "modelo_autogluon_binario"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Coluna final que será usada pelo AutoGluon como rótulo
LABEL_FINAL = "label_binary"

# Reprodutibilidade
RANDOM_STATE = 42

# Divisão dos dados
TEST_SIZE = 0.20

# Leitura por blocos para evitar estouro de memória
CHUNKSIZE = 200000

# Quantidade máxima de amostras de linhas por classe.
# Use None para usar todo o dataset disponível.
MAX_ROWS_PER_CLASS = 200000

# Limitar número de arquivos apenas para testes rápidos.
# Usar None para processar todos os arquivos CSV do dataset.
MAX_FILES = None

# Tempo máximo de treinamento em segundos.
# 3600 = 1 hora. Use None para não definir limite.
TIME_LIMIT = 3600

#===================================
#TIPOS DE TREINAMENTO
#===================================
# Preset do AutoGluon.
# Opções comuns:
# "medium_quality_faster_train"
# "good_quality"
# "high_quality"
# "best_quality"
PRESETS = "medium_quality_faster_train"

#======================================================================================================================================
# | Preset                          | Significado                                 | Quando usar                                 |
#| ------------------------------- | ------------------------------------------- | ------------------------------------------- |
#| `"medium_quality_faster_train"` | Treinamento mais rápido, qualidade moderada | Testes iniciais, validação do pipeline      |
#| `"good_quality"`                | Melhor equilíbrio entre tempo e desempenho  | Treinamento intermediário                   |
#| `"high_quality"`                | Mais modelos e melhor desempenho            | Quando já confirmou que o pipeline funciona |
#| `"best_quality"`                | Busca melhor qualidade possível             | Treinamento final, com mais tempo e memória |
#=========================================================================================================================================




# Se True, apaga o modelo anterior antes de treinar novamente.
# Use com cuidado.
APAGAR_MODELO_ANTERIOR = False # Define se o notebook deve apagar ou não modelo AutoGluon que já existe

print("Configurações definidas.")
print(f"Dataset : {DATASET_DIR}")
print(f"Saída   : {RESULTS_DIR}")
print(f"Modelo  : {MODEL_DIR}")


Configurações definidas.
Dataset : C:\Dataset\CSV
Saída   : C:\Dataset\Resultados\AutoGluon_Binario
Modelo  : C:\Dataset\Resultados\AutoGluon_Binario\modelo_autogluon_binario


In [3]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================


# Detecta automaticamente a coluna de rótulo existente no dataset
def detectar_coluna_rotulo(columns):
    """
    Detecta automaticamente a coluna de rótulo, caso exista no CSV.
    """
#==========================================================================
# Lista com possíveis nomes usados para representar a coluna de rótulo.
# A coluna de rótulo é a coluna que contém a classe de cada amostra,
# por exemplo: BenignTraffic, DDoS, DoS, Mirai, Recon etc.
#
# Como diferentes datasets podem usar nomes diferentes para essa coluna,
# o código testa várias possibilidades.
#=========================================================================
    candidatos = [
        "label", "Label",
        "Attack", "attack",
        "Class", "class",
        "category", "Category",
        "label_multiclass",
        "label_grouped",
        "label_binary"
    ]

    for col in candidatos:
        if col in columns:
            return col

    return None


def converter_para_binario(valor):
    """
    Converte qualquer rótulo do CICIoT2023 para o cenário binário:
    - Benign
    - Malicious
    """
    valor = str(valor).strip().lower()

    if "benign" in valor:
        return "Benign"
    else:
        return "Malicious"


def remover_colunas_de_rotulo_antigas(df, label_final=LABEL_FINAL):
    """
    Remove colunas antigas de rótulo para evitar vazamento de informação.
    """
    colunas_remover = [
        "label", "Label",
        "Attack", "attack",
        "Class", "class",
        "category", "Category",
        "label_multiclass",
        "label_grouped",
        "source_file",
        "Source_File",
        "filename",
        "Filename"
    ]

    colunas_remover = [
        col for col in colunas_remover
        if col in df.columns and col != label_final
    ]

    return df.drop(columns=colunas_remover, errors="ignore")


def limpar_dataset(df, label_final=LABEL_FINAL):
    """
    Aplica a limpeza básica antes do treinamento:
    - remove colunas duplicadas;
    - remove linhas sem rótulo;
    - troca infinitos por NaN;
    - remove colunas totalmente vazias;
    - remove colunas constantes.
    """
    df = df.copy()

    # Remover colunas duplicadas
    df = df.loc[:, ~df.columns.duplicated()]

    # Remover linhas sem rótulo
    df = df.dropna(subset=[label_final])

    # Substituir infinitos por NaN em colunas numéricas
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

    # Remover colunas totalmente vazias
    df = df.dropna(axis=1, how="all")

    # Remover colunas constantes, exceto o rótulo
    feature_cols = [col for col in df.columns if col != label_final]

    const_cols = [
        col for col in feature_cols
        if df[col].nunique(dropna=False) <= 1
    ]

    df = df.drop(columns=const_cols, errors="ignore")

    return df


def limpar_para_inferencia(X):
    """
    Limpa X_test antes da predição.
    Mantém as mesmas colunas, apenas troca valores problemáticos por NaN.
    """
    X = X.copy()

    numeric_cols = X.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) > 0:
        X[numeric_cols] = X[numeric_cols].replace([np.inf, -np.inf], np.nan)

        limite_float64 = np.finfo(np.float64).max
        for col in numeric_cols:
            X.loc[X[col].abs() >= limite_float64, col] = np.nan

    return X


print("Funções auxiliares criadas.")


Funções auxiliares criadas.


In [4]:
# ============================================================
# 4. LOCALIZAR ARQUIVOS CSV
# ============================================================

csv_files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.csv")))

# Evita carregar arquivo já mesclado, caso exista, para não duplicar dados.
csv_files = [
    file for file in csv_files
    if "merged" not in os.path.basename(file).lower()
]

if MAX_FILES is not None:
    csv_files = csv_files[:MAX_FILES]

print(f"Total de arquivos CSV encontrados: {len(csv_files)}")

if len(csv_files) == 0:
    raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em: {DATASET_DIR}")

print("\nPrimeiros arquivos encontrados:")
for file in csv_files[:10]:
    print("-", os.path.basename(file))


Total de arquivos CSV encontrados: 309

Primeiros arquivos encontrados:
- Backdoor_Malware.pcap.csv
- BenignTraffic.pcap.csv
- BenignTraffic1.pcap.csv
- BenignTraffic2.pcap.csv
- BenignTraffic3.pcap.csv
- BrowserHijacking.pcap.csv
- CommandInjection.pcap.csv
- DDoS-ACK_Fragmentation.pcap.csv
- DDoS-ACK_Fragmentation1.pcap.csv
- DDoS-ACK_Fragmentation10.pcap.csv


In [5]:
# ============================================================
# 5. CARREGAR DATASET EM CHUNKS
# ============================================================

dados = []
contagem = {"Benign": 0, "Malicious": 0}

parar_leitura = False

for file in csv_files:
    nome_arquivo = os.path.basename(file)
    classe_arquivo = converter_para_binario(nome_arquivo)

    print(f"\nLendo arquivo: {nome_arquivo}")

    try:
        for chunk in pd.read_csv(file, chunksize=CHUNKSIZE, low_memory=False):
            # Padronizar nomes das colunas
            chunk.columns = chunk.columns.astype(str).str.strip()

            # Detectar rótulo, se existir
            label_original = detectar_coluna_rotulo(chunk.columns)

            if label_original is not None:
                chunk[LABEL_FINAL] = chunk[label_original].apply(converter_para_binario)
            else:
                chunk[LABEL_FINAL] = classe_arquivo

            # Remover rótulos antigos para evitar vazamento
            chunk = remover_colunas_de_rotulo_antigas(chunk, LABEL_FINAL)

            # Controlar quantidade máxima por classe
            if MAX_ROWS_PER_CLASS is not None:
                partes = []

                for classe in ["Benign", "Malicious"]:
                    restante = MAX_ROWS_PER_CLASS - contagem[classe]

                    if restante <= 0:
                        continue

                    sub = chunk[chunk[LABEL_FINAL] == classe]

                    if len(sub) > restante:
                        sub = sub.sample(
                            n=restante,
                            random_state=RANDOM_STATE
                        )

                    contagem[classe] += len(sub)

                    if len(sub) > 0:
                        partes.append(sub)

                if len(partes) > 0:
                    dados.append(pd.concat(partes, ignore_index=True))

                print("Contagem parcial:", contagem)

                if all(contagem[c] >= MAX_ROWS_PER_CLASS for c in contagem):
                    parar_leitura = True
                    break

            else:
                dados.append(chunk)

        if parar_leitura:
            break

    except Exception as e:
        print(f"Erro ao ler {nome_arquivo}: {e}")

if len(dados) == 0:
    raise ValueError("Nenhum dado foi carregado. Verifique os arquivos CSV.")

df = pd.concat(dados, ignore_index=True)

print("\nDataset carregado com sucesso!")
print("Shape inicial:", df.shape)

print("\nDistribuição inicial das classes:")
print(df[LABEL_FINAL].value_counts())



Lendo arquivo: Backdoor_Malware.pcap.csv
Contagem parcial: {'Benign': 0, 'Malicious': 3218}

Lendo arquivo: BenignTraffic.pcap.csv
Contagem parcial: {'Benign': 200000, 'Malicious': 3218}
Contagem parcial: {'Benign': 200000, 'Malicious': 3218}

Lendo arquivo: BenignTraffic1.pcap.csv
Contagem parcial: {'Benign': 200000, 'Malicious': 3218}
Contagem parcial: {'Benign': 200000, 'Malicious': 3218}

Lendo arquivo: BenignTraffic2.pcap.csv
Contagem parcial: {'Benign': 200000, 'Malicious': 3218}
Contagem parcial: {'Benign': 200000, 'Malicious': 3218}

Lendo arquivo: BenignTraffic3.pcap.csv
Contagem parcial: {'Benign': 200000, 'Malicious': 3218}

Lendo arquivo: BrowserHijacking.pcap.csv
Contagem parcial: {'Benign': 200000, 'Malicious': 9077}

Lendo arquivo: CommandInjection.pcap.csv
Contagem parcial: {'Benign': 200000, 'Malicious': 14486}

Lendo arquivo: DDoS-ACK_Fragmentation.pcap.csv
Contagem parcial: {'Benign': 200000, 'Malicious': 39576}

Lendo arquivo: DDoS-ACK_Fragmentation1.pcap.csv
Conta

In [6]:
# ============================================================
# 6. TRATAR E VALIDAR DATASET
# ============================================================

df = limpar_dataset(df, LABEL_FINAL)

print("\nShape após limpeza:", df.shape)

print("\nDistribuição das classes após limpeza:")
print(df[LABEL_FINAL].value_counts())

classes = sorted(df[LABEL_FINAL].unique())

if len(classes) < 2:
    raise ValueError(
        f"O dataset contém apenas uma classe: {classes}. "
        "Para classificação binária, é necessário ter Benign e Malicious."
    )

print("\nClasses detectadas:", classes)

# Opcional: salvar um resumo do dataset tratado
resumo_classes_path = RESULTS_DIR / "distribuicao_classes_binario.csv"
df[LABEL_FINAL].value_counts().to_csv(resumo_classes_path, header=["quantidade"])

print(f"\nResumo da distribuição salvo em: {resumo_classes_path}")



Shape após limpeza: (400000, 40)

Distribuição das classes após limpeza:
label_binary
Malicious    200000
Benign       200000
Name: count, dtype: int64

Classes detectadas: ['Benign', 'Malicious']

Resumo da distribuição salvo em: C:\Dataset\Resultados\AutoGluon_Binario\distribuicao_classes_binario.csv


In [7]:
# ============================================================
# 7. DIVIDIR EM TREINO E TESTE
# ============================================================

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_FINAL]
)

train_data = TabularDataset(train_df)
test_data = TabularDataset(test_df)

print("Divisão concluída.")
print("Treino:", train_data.shape)
print("Teste :", test_data.shape)

print("\nClasses no treino:")
print(train_data[LABEL_FINAL].value_counts())

print("\nClasses no teste:")
print(test_data[LABEL_FINAL].value_counts())


Divisão concluída.
Treino: (320000, 40)
Teste : (80000, 40)

Classes no treino:
label_binary
Malicious    160000
Benign       160000
Name: count, dtype: int64

Classes no teste:
label_binary
Malicious    40000
Benign       40000
Name: count, dtype: int64


In [8]:
# ============================================================
# 8. TREINAR MODELO AUTOGLUON
# ============================================================

if APAGAR_MODELO_ANTERIOR and MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
    print(f"Modelo anterior removido de: {MODEL_DIR}")

predictor = TabularPredictor(
    label=LABEL_FINAL,
    path=str(MODEL_DIR),
    problem_type="binary",
    eval_metric="balanced_accuracy",
    verbosity=2
).fit(
    train_data=train_data,
    presets=PRESETS,
    time_limit=TIME_LIMIT
)

print("\nTreinamento finalizado.")
print(f"Modelo salvo em: {MODEL_DIR}")


Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       4.86 GB / 15.88 GB (30.6%)
Disk Space Avail:   208.33 GB / 930.66 GB (22.4%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "C:\Dataset\Resultados\AutoGluon_Binario\modelo_autogluon_binario"
Train Data Rows:    320000
Train Data Columns: 39
Label Column:       label_binary
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = Malicious, class 0 = Benign
	Note: For your binary classification, AutoG


Treinamento finalizado.
Modelo salvo em: C:\Dataset\Resultados\AutoGluon_Binario\modelo_autogluon_binario


In [ ]:
# ============================================================
# 9. PREDIÇÃO E AVALIAÇÃO AUTOGLUON
# ============================================================

X_test = test_data.drop(columns=[LABEL_FINAL])
y_test = test_data[LABEL_FINAL].copy()

X_test = limpar_para_inferencia(X_test)

y_pred = predictor.predict(X_test)

print("\nPrimeiras predições:")
print(y_pred.head())

print("\n====================================================")
print("AVALIAÇÃO AUTOGLUON")
print("====================================================")

avaliacao_autogluon = predictor.evaluate(test_data)
print(avaliacao_autogluon)


In [ ]:
# ============================================================
# 10. MÉTRICAS DETALHADAS COM SCIKIT-LEARN
# ============================================================

labels_order = ["Benign", "Malicious"]

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels_order
)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    pos_label="Malicious",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label="Malicious",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label="Malicious",
    zero_division=0
)

specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
mcc = matthews_corrcoef(y_test, y_pred)

false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0

roc_auc = np.nan
avg_precision = np.nan
y_proba = None

try:
    y_proba = predictor.predict_proba(X_test)

    if "Malicious" in y_proba.columns:
        y_score = y_proba["Malicious"]
        y_test_bin = y_test.map({"Benign": 0, "Malicious": 1})

        roc_auc = roc_auc_score(y_test_bin, y_score)
        avg_precision = average_precision_score(y_test_bin, y_score)

except Exception as e:
    print(f"Não foi possível calcular ROC-AUC/Average Precision: {e}")

print("\n====================================================")
print("RESULTADOS - AUTOGLUON - CLASSIFICAÇÃO BINÁRIA")
print("====================================================")
print(f"Accuracy              : {accuracy:.6f}")
print(f"Balanced Accuracy     : {balanced_acc:.6f}")
print(f"Precision             : {precision:.6f}")
print(f"Recall / Sensibilidade: {recall:.6f}")
print(f"F1-score              : {f1:.6f}")
print(f"Specificity           : {specificity:.6f}")
print(f"ROC-AUC               : {roc_auc:.6f}")
print(f"Average Precision     : {avg_precision:.6f}")
print(f"MCC                   : {mcc:.6f}")
print(f"False Positive Rate   : {false_positive_rate:.6f}")
print(f"False Negative Rate   : {false_negative_rate:.6f}")


In [ ]:
# ============================================================
# 11. RELATÓRIO, MATRIZ DE CONFUSÃO E LEADERBOARD
# ============================================================

print("\n====================================================")
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("====================================================")

report_text = classification_report(
    y_test,
    y_pred,
    labels=labels_order,
    zero_division=0
)

print(report_text)

report_dict = classification_report(
    y_test,
    y_pred,
    labels=labels_order,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()

print("\n====================================================")
print("MATRIZ DE CONFUSÃO")
print("====================================================")
print(cm)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels_order
)
disp.plot(values_format="d", cmap="Blues")
plt.title("Matriz de Confusão - AutoGluon Binário")
plt.tight_layout()

cm_path = RESULTS_DIR / "matriz_confusao_autogluon_binario.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nMatriz de confusão salva em: {cm_path}")

leaderboard = predictor.leaderboard(test_data, silent=True)

print("\n====================================================")
print("LEADERBOARD - AUTOGLUON")
print("====================================================")
print(leaderboard) # Tabela de comparação dos modelos treinados pelo AutoGluon


In [ ]:
# ============================================================
# 12. SALVAR RESULTADOS EM CSV
# ============================================================

metrics_df = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall / Sensibilidade",
        "F1-score",
        "Specificity",
        "ROC-AUC",
        "Average Precision",
        "MCC",
        "False Positive Rate",
        "False Negative Rate"
    ],
    "Valor": [
        accuracy,
        balanced_acc,
        precision,
        recall,
        f1,
        specificity,
        roc_auc,
        avg_precision,
        mcc,
        false_positive_rate,
        false_negative_rate
    ]
})

metrics_path = RESULTS_DIR / "metricas_autogluon_binario.csv"
report_path = RESULTS_DIR / "classification_report_autogluon_binario.csv"
leaderboard_path = RESULTS_DIR / "leaderboard_autogluon_binario.csv"

metrics_df.to_csv(metrics_path, index=False)
report_df.to_csv(report_path)
leaderboard.to_csv(leaderboard_path, index=False)

print("\nArquivos CSV salvos:")
print(metrics_path)
print(report_path)
print(leaderboard_path)


In [ ]:
# ============================================================
# 13. SALVAR MÉTRICAS E LEADERBOARD EM PNG
# ============================================================

# Métricas em PNG
fig, ax = plt.subplots(figsize=(9, 4))
ax.axis("off")

metrics_table = metrics_df.copy()
metrics_table["Valor"] = metrics_table["Valor"].apply(
    lambda x: f"{x:.6f}" if pd.notna(x) else "NaN"
)

table = ax.table(
    cellText=metrics_table.values,
    colLabels=metrics_table.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.4)

plt.title("Métricas - AutoGluon Binário", fontsize=12)
plt.tight_layout()

metrics_png_path = RESULTS_DIR / "metricas_autogluon_binario.png"
plt.savefig(metrics_png_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Métricas salvas em PNG: {metrics_png_path}")


# Leaderboard em PNG
leaderboard_view = leaderboard.head(20).copy()

for col in leaderboard_view.columns:
    if pd.api.types.is_numeric_dtype(leaderboard_view[col]):
        leaderboard_view[col] = leaderboard_view[col].round(6)

fig, ax = plt.subplots(figsize=(14, max(5, 0.45 * len(leaderboard_view))))
ax.axis("off")

table = ax.table(
    cellText=leaderboard_view.values,
    colLabels=leaderboard_view.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.3)

plt.title("Leaderboard - AutoGluon Binário", fontsize=12)
plt.tight_layout()

leaderboard_png_path = RESULTS_DIR / "leaderboard_autogluon_binario.png"
plt.savefig(leaderboard_png_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Leaderboard salvo em PNG: {leaderboard_png_path}")


In [ ]:
# ============================================================
# 14. SALVAR INFERÊNCIAS EM CSV
# ============================================================

inferencias_df = X_test.copy()
inferencias_df["classe_real"] = y_test.values
inferencias_df["classe_predita"] = y_pred.values

if y_proba is not None:
    for col in y_proba.columns:
        inferencias_df[f"prob_{col}"] = y_proba[col].values

inferencias_path = RESULTS_DIR / "inferencias_autogluon_binario.csv"
inferencias_df.to_csv(inferencias_path, index=False)

print(f"Inferências salvas em: {inferencias_path}")


In [ ]:
# ============================================================
# 15. FINALIZAÇÃO
# ============================================================

print("\n====================================================")
print("PROCESSO FINALIZADO COM SUCESSO")
print("====================================================")
print(f"Modelo salvo em     : {MODEL_DIR}")
print(f"Resultados salvos em: {RESULTS_DIR}")
